# Glue Catalog Update — getSink + DynamicFrame

## The Problem

You have CSV files landing in S3 without partitions.
You want to write them as partitioned Parquet to another S3 location.

```
Source (flat, no partitions):
s3://sanjay-de-bucket-2026/orders_raw/
    orders_day1.csv
    orders_day2.csv
    orders_day3.csv

Target (partitioned by order_date):
s3://sanjay-de-bucket-2026/orders_processed/
    order_date=2013-07-25/
    order_date=2013-07-26/
    order_date=2013-07-27/
```

**The question is: after writing, can Athena see the new partitions immediately?**

```
With df.write (pure Spark):
  Data lands in S3 but Glue Catalog is NOT updated
  Athena query returns no data
  Fix: run crawler OR run MSCK REPAIR TABLE manually

With getSink + enableUpdateCatalog=True:
  Data lands in S3 AND partitions registered in Glue Catalog
  Athena query returns data immediately — no crawler needed
```

---
## Table of Contents

- [Setup](#Setup)
- [Read Input Data](#Read-Input-Data)
- [Approach 1 - Pure Spark df.write](#Approach-1---Pure-Spark-df.write)
- [Approach 2 - getSink with enableUpdateCatalog](#Approach-2---getSink-with-enableUpdateCatalog)
- [Who owns what](#Who-owns-what)
- [updateBehavior options](#updateBehavior-options)
- [Approach 3 - write_dynamic_frame.from_catalog with enableUpdateCatalog](#Approach-3---write_dynamic_frame.from_catalog-with-enableUpdateCatalog)
- [Final Comparison](#Final-Comparison)

---
## Setup

In [ ]:
%idle_timeout 30
%glue_version 5.1
%worker_type G.1X
%number_of_workers 2

In [17]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql.functions import col, upper, current_date
from awsglue.dynamicframe import DynamicFrame
from pyspark.sql.types import StructType, StructField , IntegerType , StringType ,DoubleType

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session


print("spark session created")

S3_INPUT  = "s3://sanjay-de-bucket-2026/orders_raw/"
S3_OUTPUT = "s3://sanjay-de-bucket-2026/orders_processed/"
DATABASE  = "ecommerce_db"
TABLE     = "orders_processed"

print("Setup complete")

spark session created
Setup complete


## Read Input Data

Reading CSV files from S3. Each file has this structure:
```
order_id,order_date,order_customer_id,order_status
1,2013-07-25 00:00:00.0,11599,CLOSED
2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT
```

In [18]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(S3_INPUT)

# order_date comes in as '2013-07-25 00:00:00.0'
# extract just the date part for clean partition folder names
# order_date=2013-07-25  instead of  order_date=2013-07-25 00:00:00.0
from pyspark.sql.functions import to_date
df = df.withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd HH:mm:ss.S"))

df.printSchema()
df.show(5)

root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)

+--------+----------+-----------------+---------------+
|order_id|order_date|order_customer_id|   order_status|
+--------+----------+-----------------+---------------+
|     105|2013-07-26|             8220|       COMPLETE|
|     106|2013-07-26|              395|     PROCESSING|
|     107|2013-07-26|             1845|       COMPLETE|
|     108|2013-07-26|            12149|     PROCESSING|
|     109|2013-07-26|             9345|PENDING_PAYMENT|
+--------+----------+-----------------+---------------+
only showing top 5 rows



**Expected Output:**
```
root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)      <-- clean date after conversion
 |-- order_customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)

+--------+----------+-----------------+---------------+
|order_id|order_date|order_customer_id|   order_status|
+--------+----------+-----------------+---------------+
|       1|2013-07-25|            11599|         CLOSED|
|       2|2013-07-25|              256|PENDING_PAYMENT|
|       3|2013-07-25|            12111|       COMPLETE|
|       4|2013-07-25|             8827|         CLOSED|
|       5|2013-07-25|            11318|       COMPLETE|
+--------+----------+-----------------+---------------+
```

---
## Approach 1 - Pure Spark df.write

This is what most people do by default.

In [ ]:
df.write \
    .mode("overwrite") \
    .partitionBy("order_date") \
    .parquet(S3_OUTPUT)

In [ ]:
#If you want both S3 path + catalog table
df.write \
    .mode("overwrite") \
    .option("path", S3_OUTPUT) \
    .partitionBy("order_date") \
    .saveAsTable("ecommerce_db.example_orders")

26/05/05 17:00:51 INFO HiveConf: Found configuration file file:/home/glue_user/spark/conf/hive-site.xml
26/05/05 17:00:56 WARN EC2MetadataUtils: Unable to retrieve the requested metadata (/latest/dynamic/instance-identity/document). connecting to 169.254.169.254:80: connecting to 169.254.169.254:80: dial tcp 169.254.169.254:80: connectex: A socket operation was attempted to an unreachable network. (Service: null; Status Code: 403; Error Code: null; Request ID: null; Proxy: null)
com.amazonaws.AmazonServiceException: connecting to 169.254.169.254:80: connecting to 169.254.169.254:80: dial tcp 169.254.169.254:80: connectex: A socket operation was attempted to an unreachable network. (Service: null; Status Code: 403; Error Code: null; Request ID: null; Proxy: null)
	at com.amazonaws.internal.EC2ResourceFetcher.handleErrorResponse(EC2ResourceFetcher.java:161)
	at com.amazonaws.internal.EC2ResourceFetcher.doReadResource(EC2ResourceFetcher.java:106)
	at com.amazonaws.internal.EC2ResourceFetc

**What happened:**
```
S3:
  s3://sanjay-de-bucket-2026/orders_processed/order_date=2013-07-25/part-00000.parquet  <- written
  s3://sanjay-de-bucket-2026/orders_processed/order_date=2013-07-26/part-00000.parquet  <- written
  ...

Glue Catalog:
  ecommerce_db.orders_silver  <- NOT updated, partitions not registered
```

Now if you query Athena:
```sql
SELECT * FROM ecommerce_db.orders_silver WHERE order_date = '2013-07-25'
-- Returns 0 rows  <-- Athena does not know this partition exists
```

**To fix this you have two options — both are manual:**
```sql
-- Option A: repair table
MSCK REPAIR TABLE ecommerce_db.orders_silver;

-- Option B: add partition explicitly
ALTER TABLE ecommerce_db.orders_silver
ADD PARTITION (order_date='2013-07-25')
LOCATION 's3://sanjay-de-bucket-2026/orders_processed/order_date=2013-07-25/';
```

> In a daily pipeline this means you need an extra step after every write just to keep Athena in sync.

---
## Approach 2 - getSink with enableUpdateCatalog

**Key point:** `getSink` is a method on `glueContext`, not on DynamicFrame.
But `writeFrame()` only accepts a DynamicFrame — so you must convert your DataFrame first.

```
GlueContext  →  provides getSink() + catalog update capability
DynamicFrame →  required input format for writeFrame()
```

In [ ]:
# Step 1 - convert DataFrame to DynamicFrame
# writeFrame() only accepts DynamicFrame, not DataFrame
dyf = DynamicFrame.fromDF(df, glueContext, "orders_output")

# Step 2 - configure the sink
sink = glueContext.getSink(
    connection_type="s3",
    path=S3_OUTPUT,
    enableUpdateCatalog=True,           # <-- this is the key flag
    updateBehavior="UPDATE_IN_DATABASE", # adds new partitions + updates table metadata
    partitionKeys=["order_date"]       # partition by order_date
)

# Step 3 - tell Glue which catalog table to update
sink.setCatalogInfo(
    catalogDatabase=DATABASE,
    catalogTableName=TABLE
)

# Step 4 - set output format
# NOTE: use 'glueparquet' not 'parquet' — getSink uses Glue's own format names
sink.setFormat("parquet")

# Step 5 - write
sink.writeFrame(dyf)

26/05/05 11:16:02 WARN Job$: Job run ID 1637f6db-b5ad-46b1-a268-b1535349eb92 is either null or empty or its same as Job name. 
26/05/05 11:16:03 WARN LakeformationRetryWrapper$: Cannot find LakeformationClientWrapper, Lakeformation is not supported in local development mode.


**What happened:**
```
S3:
  s3://sanjay-de-bucket-2026/orders_processed/order_date=2013-07-25/part-00000.parquet
  s3://sanjay-de-bucket-2026/orders_processed/order_date=2013-07-26/part-00000.parquet
  ...

Glue Catalog:
  ecommerce_db.orders_silver partitions: 2013-07-25, 2013-07-26, ...  <- updated automatically
```

Now Athena can query immediately:
```sql
SELECT * FROM ecommerce_db.orders_silver WHERE order_date = '2013-07-25'
-- Returns results immediately, no MSCK REPAIR needed
```

---
## Who owns what

This is an important distinction to understand clearly:

```
glueContext.getSink()     <- GlueContext owns this
                             enableUpdateCatalog is a GlueContext feature
                             nothing to do with DynamicFrame

sink.writeFrame(dyf)      <- writeFrame() requires a DynamicFrame as input
                             you cannot pass a Spark DataFrame here directly
                             this is where DynamicFrame is needed
```

So the accurate statement is:
- **Catalog update** is a **GlueContext** capability
- **DynamicFrame** is the required data format to use it
- Pure Spark `df.write` has no path to this feature — you need both

---
## updateBehavior options

| Value | What it does |
|---|---|
| `UPDATE_IN_DATABASE` | Adds new partitions and updates existing table metadata in catalog |
| `LOG` | Only logs what would change, does NOT update catalog — useful for testing |

```python
# Use LOG first to verify what will be updated before actually updating
sink = glueContext.getSink(
    connection_type="s3",
    path=S3_OUTPUT,
    enableUpdateCatalog=True,
    updateBehavior="LOG",  # dry run — check logs, nothing written to catalog
    partitionKeys=["order_date"]
)
```

---
## Approach 3 - write_dynamic_frame.from_catalog with enableUpdateCatalog

Unlike `getSink`, this works in the local developer environment too.
The target table must already exist in Glue Catalog before running this.

In [ ]:
# Read from S3 using DynamicFrame directly
from pyspark.sql.functions import to_date

dyf_raw = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": [S3_INPUT], "recurse": True},
    format="csv",
    format_options={"withHeader": True}
)

# Convert to DataFrame to apply the date transformation
df_raw = dyf_raw.toDF()
df_raw = df_raw.withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd HH:mm:ss.S"))

df_raw.printSchema()
df_raw.show(5)

**Expected Output:**
```
root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)      <-- clean date after conversion
 |-- order_customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+----------+-----------------+---------------+
|order_id|order_date|order_customer_id|   order_status|
+--------+----------+-----------------+---------------+
|       1|2013-07-25|            11599|         CLOSED|
|       2|2013-07-25|              256|PENDING_PAYMENT|
|       3|2013-07-25|            12111|       COMPLETE|
+--------+----------+-----------------+---------------+
```

In [ ]:
# Convert back to DynamicFrame
dyf_output = DynamicFrame.fromDF(df_raw, glueContext, "orders_output")

# Write back to catalog table and update partitions automatically
# Target table must already exist in Glue Catalog
glueContext.write_dynamic_frame.from_catalog(
    frame=dyf_output,
    database=DATABASE,
    table_name=TABLE,
    additional_options={
        "enableUpdateCatalog": True,
        "partitionKeys": ["order_date"],
        "updateBehavior": "UPDATE_IN_DATABASE"
    }
)

**What happened:**
```
S3:
  s3://sanjay-de-bucket-2026/orders_processed/order_date=2013-07-25/part-00000.parquet
  s3://sanjay-de-bucket-2026/orders_processed/order_date=2013-07-26/part-00000.parquet
  ...

Glue Catalog:
  ecommerce_db.orders_processed partitions: 2013-07-25, 2013-07-26, ...  <- updated automatically
```

Athena can query immediately:
```sql
SELECT * FROM ecommerce_db.orders_processed WHERE order_date = '2013-07-25'
-- Returns results immediately, no MSCK REPAIR needed
```

---
## Final Comparison

| | Writes to S3 | Updates Glue Catalog | Works Locally | Requires DynamicFrame | Notes |
|---|---|---|---|---|---|
| `df.write.parquet()` | ✅ | ❌ | ✅ | ❌ | Catalog needs manual MSCK REPAIR |
| `df.write.saveAsTable()` | ✅ | ✅ | ✅ | ❌ | Simplest option, needs Glue catalog as Hive metastore enabled |
| `write_dynamic_frame.from_options` | ✅ | ❌ | ✅ | ✅ | Catalog needs manual MSCK REPAIR |
| `write_dynamic_frame.from_catalog` + `enableUpdateCatalog` | ✅ | ✅ | ✅ | ✅ | Target table must already exist in catalog |
| `getSink` + `enableUpdateCatalog=True` | ✅ | ✅ | ❌ | ✅ | Only works on real AWS Glue infra, not local dev |

## When to use which

| Scenario | Best Choice |
|---|---|
| Write to S3, run crawler after | `df.write.parquet()` — simplest |
| Write to S3 + update catalog, pure Spark | `df.write.saveAsTable()` — simplest, no DynamicFrame needed |
| Write to S3 + update catalog, Glue API | `write_dynamic_frame.from_catalog` + `enableUpdateCatalog` |
| Write to Redshift or RDS via Glue Connection | `getSink` — only option for non-S3 targets |
| Write to Redshift large data (bulk load) | `getSink` — uses S3 staging + COPY command, much faster than JDBC |
| Write to PostgreSQL / SQL Server / MySQL | Native Spark JDBC — `df.write.format("jdbc")` is equally capable |
| Write to Snowflake | Native Snowflake Spark connector — `getSink` has no advantage |
| Job Bookmark needed | `write_dynamic_frame` + `job.commit()` |
